In [1]:
import sys
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
import config
from rx import io, text, places, remote

| Step | Purpose |
|---|---|
| 3.1 | File type from the title (unchanged) |
| 3.2 | Name, served state (unchanged) + **all claim numbers + bounty-land acres** |
| 3.3 | `decl_text`: lowercase + whitespace; **who is declaring** (veteran/widow); **application year** |
| 3.4 | Assertions + save |

In [2]:
import pandas as pd
files = io.read_stage("interim", "files_raw")
t = files.title

files["ftype"] = "other"
files.loc[t.str.contains(r"File\s+(?:B\.?\s*L|Bl\.?\s*Wt|Wt\b)"), "ftype"] = "BLWT"
for code in ["S", "W", "R"]:
    files.loc[t.str.contains(rf"File\s+{code}(?![a-z])"), "ftype"] = code
files.loc[t.str.contains(r"File\s+(?:Old War|O\.\s*W)"), "ftype"] = "old_war"
files.loc[t.str.contains(r"File\s+\[(?:blank|illegible)\]", case=False), "ftype"] = "number_blank"
files.loc[t.str.endswith("Application File"), "ftype"] = "roll_header"
files.loc[t.str.contains("Administrative Sheets"), "ftype"] = "admin_sheet"
files.loc[~t.str.contains("Application File"), "ftype"] = "single_document"
files.loc[t.str.contains(r"\b[NM]ot\.?\s+Rev", case=False), "ftype"] = "not_rev"
print(files.ftype.value_counts())


ftype
S                  36105
W                  25599
R                  11145
BLWT                3227
roll_header         1445
number_blank         622
admin_sheet          303
other                181
not_rev               54
single_document       36
old_war               28
Name: count, dtype: int64


In [4]:
head = (t.str.replace(r"^.*?Application File\s*", "", regex=True)
         .str.split(r",\s*(?:for\s+)?[A-Z][a-z]", n=1, regex=True).str[0])
CLAIM_RX = r"(?<![A-Za-z.])([SWR])(?![a-z])\s*[.:;\-]?\s*(\d[\d,]*)"
files["claim_nos"] = head.str.findall(CLAIM_RX).map(
    lambda xs: [f"{a}{b.replace(',', '')}" for a, b in xs])
files["n_claims"] = files.claim_nos.str.len()

BLW_RX = (r"(?:B\s*\.?\s*L\.?\s*W\.?\s*T?t?|BLWT?|Bl\.?\s*Wt|(?<![A-Za-z])Wt)\.?\s*"
          r"([\d,]+)\s*[-_=]\s*(\d+)(?:\s*[-_=]\s*(\d{2,4}))?")
blw = t.str.extract(BLW_RX)
acres = pd.to_numeric(blw[1], errors="coerce")
act = pd.to_numeric(blw[2], errors="coerce")
is_year = acres >= 1776                                  # "Wt. 123-1855": the 2nd number is the Act
act = act.where(~is_year, acres)
acres = acres.where(~is_year)
act = act.where(act >= 100, act + 1800)                  # 55 → 1855, 50 → 1850
files["blw_no"] = blw[0].str.replace(",", "", regex=False)
files["blw_acres"] = acres.astype("Int64")
files["blw_act"] = act.astype("Int64")
files["bl_rejected"] = t.str.contains(
    r"B\.?\s*L\.?\s*Re[gj]|Rej\.?\s*B\.?\s*L|Rejected B\.?\s*L|Bounty Land Register|Bl\.? Reg")
print(files.blw_acres.notna().sum(), files.blw_act.value_counts(), files.bl_rejected.sum())

parts = t.str.split(",")
files["served_raw"] = parts.str[-1].str.strip()
files["veteran_name"] = parts.str[-2].str.strip().str.replace(r"^for\s+", "", regex=True)
files.loc[parts.str.len() < 3, ["served_raw", "veteran_name"]] = pd.NA
files["served_state"] = places.normalise_state_series(files.served_raw.str.lower())
print(files.n_claims.value_counts().sort_index(), files.blw_acres.value_counts().head())

3068 blw_act
1855    304
1850     19
1864      1
1812      1
1858      1
1792      1
Name: count, dtype: Int64 71
n_claims
0     6162
1    72530
2       52
3        1
Name: count, dtype: int64 blw_acres
100    959
300    648
200    635
160    319
400    173
Name: count, dtype: Int64


In [5]:
d = io.read_stage("interim", "pages_decl")
d["text"] = text.normalise_whitespace(d.text.str.lower())
d = d[d.text.notna()].sort_values(["NAID", "pageNumber"]).reset_index(drop=True)

head = d.offs.eq(0)
d["person"] = pd.Series(pd.NA, index=d.index, dtype="string")
d.loc[head, "person"] = d.loc[head, "text"].str.contains(r"\bwidow\b").map({True: "widow", False: "veteran"})
d["person"] = d.groupby("NAID").person.ffill().fillna("veteran")

yrs = d.loc[head, "text"].str.extractall(r"\b(18[1-6]\d)\b")[0].astype(int)
d["app_year"] = yrs[yrs >= 1818].groupby(level=0).max().astype("Int64")
print(d.person.value_counts(), d.loc[head, "app_year"].describe())

person
veteran    223582
widow      110786
Name: count, dtype: int64[pyarrow] count       111706.0
mean     1834.465078
std         10.00626
min           1818.0
25%           1832.0
50%           1832.0
75%           1842.0
max           1869.0
Name: app_year, dtype: Float64


In [6]:
assert files.NAID.is_unique and files.ftype.notna().all()
assert not d.duplicated(["NAID", "pageNumber"]).any()
assert d.NAID.isin(files.NAID).all() and d.person.notna().all()
io.write_stage(files, "interim", "files")
io.write_stage(d, "interim", "decl_text")

PosixPath('/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/interim/decl_text.parquet')